In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import ipywidgets as widgets
from IPython.display import display, HTML

# 1. Cargar datos locales usando rutas relativas
# Retrocedemos una carpeta (../) desde 'notebooks' hacia 'datasets'
df_train = pd.read_csv('../datasets/train.csv')
df_test = pd.read_csv('../datasets/test.csv')

print(f"Datos históricos (Entrenamiento): {df_train.shape[0]} registros")
print(f"Datos Mundial 2026 (Predicción): {df_test.shape[0]} equipos")

Datos históricos (Entrenamiento): 192 registros
Datos Mundial 2026 (Predicción): 48 equipos


In [3]:
display(df_train.head())

,version,team,continent,is_host,goals_scored_last_4y,goals_received_last_4y,wins_last_4y,losses_last_4y,draws_last_4y,world_cup_titles_before,...,world_cup_participations_before,groups_passed_before,round16_before,quarterfinals_before,semifinals_before,finals_before,winner,finalist,semi_finalist,quarter_finalist
0,2006,Angola,Africa,0,61,49,19,13,14,0,...,0,0,0,0,0,0,0,0,0,0
1,2006,Argentina,South America,0,97,55,31,10,10,2,...,13,10,5,6,4,4,0,0,0,1
2,2006,Australia,Oceania,0,101,34,23,8,5,0,...,1,0,0,0,0,0,0,0,0,0
3,2006,Brazil,South America,0,117,47,30,9,17,5,...,17,15,7,11,9,6,0,0,0,1
4,2006,Costa Rica,North America,0,89,84,26,25,11,0,...,2,1,1,0,0,0,0,0,0,0


In [4]:
display(df_test.head())

,version,team,continent,is_host,goals_scored_last_4y,goals_received_last_4y,wins_last_4y,losses_last_4y,draws_last_4y,world_cup_titles_before,...,world_cup_participations_before,groups_passed_before,round16_before,quarterfinals_before,semifinals_before,finals_before,winner,finalist,semi_finalist,quarter_finalist
0,2026,France,Europe,0,85,32,25,6,7,2,...,16,10,8,9,7,4,NaN,NaN,NaN,NaN
1,2026,Spain,Europe,0,104,32,29,2,8,1,...,16,11,9,6,2,1,NaN,NaN,NaN,NaN
2,2026,Argentina,South America,0,80,14,30,4,3,3,...,18,15,10,10,6,6,NaN,NaN,NaN,NaN
3,2026,England,Europe,0,82,23,26,6,7,1,...,16,13,8,10,3,1,NaN,NaN,NaN,NaN
4,2026,Portugal,Europe,0,98,31,26,5,7,0,...,8,5,4,3,2,0,NaN,NaN,NaN,NaN


In [5]:
# 1. Variables predictoras basadas en tus columnas
features = ['fifa_points_pre_tournament', 'goals_scored_last_4y', 'wins_last_4y', 'world_cup_titles_before']

# Rellenar valores nulos por si acaso
X_train = df_train[features].fillna(0)
X_test = df_test[features].fillna(0)

# 2. Escalar los datos (Crucial para igualar magnitudes)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test) # Ojo: transform, no fit_transform

# 3. Variable Objetivo (Target Proxy)
# Usamos el Top 16 del ranking histórico como proxy de "Éxito"
y_train = (df_train['fifa_rank_pre_tournament'] <= 16).astype(int)

# 4. Entrenar el modelo
modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_train_scaled, y_train)

# 5. Predecir las probabilidades para el Mundial 2026
probabilidades = modelo.predict_proba(X_test_scaled)[:, 1] * 100
df_test['Probabilidad_Clasificar_%'] = probabilidades

print("✅ Modelo entrenado exitosamente y probabilidades 2026 calculadas.")

✅ Modelo entrenado exitosamente y probabilidades 2026 calculadas.


In [ ]:
# Función para el Dashboard Interactivo
def analizar_equipo(equipo_seleccionado):
    datos_eq = df_test[df_test['team'] == equipo_seleccionado].iloc[0]
    prob = datos_eq['Probabilidad_Clasificar_%']
    
    print("="*50)
    print(f"🏆 ANÁLISIS DE LA IA PARA: {equipo_seleccionado.upper()} 🏆")
    print("="*50)
    print(f"📈 Probabilidad de Éxito: {prob:.1f}%")
    print(f"🌍 Ranking FIFA Actual:   {datos_eq['fifa_rank_pre_tournament']}")
    print(f"⚽ Goles (Últimos 4a):     {datos_eq['goals_scored_last_4y']}")
    print(f"💰 Valor Plantilla:       {datos_eq['squad_total_market_value_eur']/1000000:.1f} Millones EUR")
    print("="*50)
    
    # Gráfico de Radar
    max_valores = [
        df_test['fifa_points_pre_tournament'].max(),
        df_test['goals_scored_last_4y'].max(),
        df_test['wins_last_4y'].max(),
        df_test['squad_total_market_value_eur'].max()
    ]
    
    valores_eq = [
        datos_eq['fifa_points_pre_tournament'] / max_valores[0] * 100,
        datos_eq['goals_scored_last_4y'] / max_valores[1] * 100,
        datos_eq['wins_last_4y'] / max_valores[2] * 100,
        datos_eq['squad_total_market_value_eur'] / max_valores[3] * 100
    ]
    
    categorias = ['Puntos FIFA', 'Goles Marcados', 'Victorias Previas', 'Valor Plantilla']
    
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=valores_eq, theta=categorias, fill='toself', name=equipo_seleccionado, line_color="blue"))
    fig.add_trace(go.Scatterpolar(r=[100]*4, theta=categorias, fill='none', name='Nivel Top Mundial', line_color="red", line_dash="dash"))
    
    fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 100])), height=400)
    fig.show()

# Crear Widget
lista_equipos = sorted(df_test['team'].unique())
menu_desplegable = widgets.Dropdown(options=lista_equipos, value='Argentina', description='Selección:')

widgets.interact(analizar_equipo, equipo_seleccionado=menu_desplegable);

interactive(children=(Dropdown(description='Selección:', index=1, options=('Algeria', 'Argentina', 'Australia'…

In [7]:
# Mostrar el Top 15 de favoritos
top_equipos = df_test.sort_values(by='Probabilidad_Clasificar_%', ascending=False).head(15)

fig_ranking = px.bar(
    top_equipos, 
    x='team', 
    y='Probabilidad_Clasificar_%',
    text_auto='.1f',
    title="Top 15 Favoritos del Mundial 2026 según Regresión Logística",
    color='Probabilidad_Clasificar_%',
    color_continuous_scale='viridis'
)

fig_ranking.update_layout(xaxis_title="Selección Nacional", yaxis_title="Probabilidad (%)")
fig_ranking.show()